In [1]:
import pandas as pd
import numpy as np



def encyclicEncoding(data):

    data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)
    data["week_sin"] = np.sin(2 * np.pi * data["epi_week"] / 52)
    data["week_cos"] = np.cos(2 * np.pi * data["epi_week"] / 52)

    return data




In [3]:
import pandas as pd
import numpy as np




def handleDataInconsistency(data):
        
    data["district"] = data["district"].ffill()
    data["month"] = data["month"].ffill()

    data["cases"] = data["cases"].interpolate(method="linear")
    data["deaths"] = data["deaths"].interpolate(method="linear")


    data = data.drop_duplicates()

    data["district"] = data["district"].str.lower().str.strip()


    data["cases"] = data["cases"].clip(lower=0)
    data["deaths"] = data["deaths"].clip(lower=0)


    data = data[(data["epi_week"] >= 1) & (data["epi_week"] <= 52)]
    data = data[(data["month_num"] >= 1) & (data["month_num"] <= 12)]


    data = data.sort_values(["district", "year", "epi_week"])


    assert data["month_num"].between(1, 12).all()
    assert data["epi_week"].between(1, 52).all()

    return data


In [7]:
import numpy as np
import pandas as pd

def lagFeatures(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["t1_cases"]=df.groupby("district")["cases"].shift(1)
    df["t2_cases"]=df.groupby("district")["cases"].shift(2)
    
    df["T2M_lag1"]=df.groupby("district")["T2M_mean"].shift(1)

    df["PRECTOTCORR_lag1"]=df.groupby("district")["PRECTOTCORR_sum"].shift(1)
    df["PRECTOTCORR_lag2"]=df.groupby("district")["PRECTOTCORR_sum"].shift(2)
    df["momentum"]=df["t1_cases"]-df["t2_cases"]
    df=df.dropna(subset=["t1_cases", "t2_cases"]).reset_index(drop=True)
    return df


In [5]:
def waterProxy(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["waterProxy"]=(df.groupby("district")["PRECTOTCORR_sum"]
                        .transform(lambda elem:elem.rolling(window=2,min_periods=1).sum()))
    
    return df


In [8]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import joblib

df = pd.read_csv('../data/processed/punjab_district_weekly_merged.csv')

# Sort
df = df.sort_values(["district", "year", "epi_week"]).reset_index(drop=True)

# Train mask — defined once, used everywhere
trainMask = df['year'].isin([2016, 2017, 2018])

# Encode district on train only
encoder = OrdinalEncoder()
encoder.fit(df[trainMask][['district']])
df['district_encoded'] = encoder.transform(df[['district']])
joblib.dump(encoder, "../models/district_encoder.pkl")

# Outbreak threshold from train only
districtThreshold = df[trainMask].groupby('district')['cases'].quantile(0.90)
df['districtThreshold'] = df['district'].map(districtThreshold) 
df['prevCases']=df.groupby("district")["cases"].shift(1)
df['isOutbreak'] = (df['prevCases'] > df['districtThreshold']).astype(int)
df = df.drop(columns=['districtThreshold','prevCases'])  # cleanup helper column

df=lagFeatures(df)
df=waterProxy(df)
df=encyclicEncoding(df)
df=handleDataInconsistency(df)
# Year-wise split
train = df[df['year'].isin([2016, 2017, 2018])].copy()
val   = df[df['year'] == 2019].copy()
test  = df[df['year'] == 2020].copy()



n = len(df)
print(f"Total rows : {n}")
print(f"Train rows : {len(train)} ({len(train)/n*100:.1f}%) — 2016-2018")
print(f"Val rows   : {len(val)}   ({len(val)/n*100:.1f}%) — 2019 outbreak")
print(f"Test rows  : {len(test)}  ({len(test)/n*100:.1f}%) — 2020")
print(f"\nOutbreak weeks in train : {train['isOutbreak'].sum()}")
print(f"Outbreak weeks in val   : {val['isOutbreak'].sum()}")
print(f"Outbreak weeks in test  : {test['isOutbreak'].sum()}")

Total rows : 8568
Train rows : 5112 (59.7%) — 2016-2018
Val rows   : 1728   (20.2%) — 2019 outbreak
Test rows  : 1728  (20.2%) — 2020

Outbreak weeks in train : 450
Outbreak weeks in val   : 644
Outbreak weeks in test  : 110


In [9]:
#Feature Engineering 
#in our case,the features are mostly the derived and engineered rows like the lag features,water proxy,encycling-features and the weather condition while our target is the no. of cases

features = [ 
"district_encoded",    # Encoded district (categorical)
"T2M_mean",          # Current temperature  
"RH2M_mean",          # Current humidity 
"PRECTOTCORR_sum",    # Current precipitation 
"WS10M_mean",        # Current wind speed 
"T2M_lag1",          # Previous week temperature (delayed effect) 
"PRECTOTCORR_lag1",   # Previous week rainfall 
"PRECTOTCORR_lag2",   # Two-week rainfall (standing water) 
"t1_cases",           # Cases last week (autoregressive) 
"t2_cases",           # Cases two weeks ago 
"waterProxy",        
"month_sin",          
"month_cos",          
"week_sin",          
"week_cos",
"momentum",
"isOutbreak"          
]

target = "cases"  # Number of dengue cases to predict

xTrain = train[features].copy()
yTrain = train[target].copy()

xTest = test[features].copy()
yTest = test[target].copy()

print(f"Feature matrix shape: {xTrain.shape}") 
print(f"target vector shape:  {yTrain.shape}")

Feature matrix shape: (5112, 17)
target vector shape:  (5112,)


In [10]:
# Check before modeling
print(xTrain.isnull().sum())

district_encoded    0
T2M_mean            0
RH2M_mean           0
PRECTOTCORR_sum     0
WS10M_mean          0
T2M_lag1            0
PRECTOTCORR_lag1    0
PRECTOTCORR_lag2    0
t1_cases            0
t2_cases            0
waterProxy          0
month_sin           0
month_cos           0
week_sin            0
week_cos            0
momentum            0
isOutbreak          0
dtype: int64


In [24]:
#Scaling the data
from sklearn.preprocessing import StandardScaler
import joblib as jl

scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(xTrain)

xTestScaled = scaler.transform(xTest)
jl.dump(scaler, "../models/scaler.pkl")

print(f"Scaler means (first 5 features): {scaler.mean_[:5]}") 
print(f"Scaler stds  (first 5 features): {scaler.scale_[:5]}")

Scaler means (first 5 features): [17.5        26.71619495 36.0387173   8.2737813   2.52306729]
Scaler stds  (first 5 features): [10.38829469  8.30447763 13.64367141 15.44411543  0.72919974]


In [25]:

severityMapping={
    0:'Low',
    1:'Medium',
    2:'High',
    3:'Critical'
}

q25=yTrain.quantile(0.25)
q75=yTrain.quantile(0.75)
q90=yTrain.quantile(0.90)

def labelSeverityCases(cases):
    if cases <= q25:
        return 0  # Low
    elif cases <= q75:
        return 1  # Medium
    elif cases <= q90:
        return 2  # High
    else:
        return 3  # Critical
    

yTrainSeverity = yTrain.apply(labelSeverityCases).values
                                              
yTestSeverity = yTest.apply(labelSeverityCases).values


In [26]:
# freezing the data pipeline for saving the data from exposure to risks and corruption
import numpy as np
train.to_csv("../data/processed/train.csv", index=False)

test.to_csv("../data/processed/test.csv", index=False)

np.save("../data/processed/xTrainScaled.npy", xTrainScaled)
np.save("../data/processed/ytrain_severity.npy", yTrainSeverity)